# **7. Validación**

En esta sección se desarrolla el proceso de **validación de modelos**, con el objetivo de evaluar de manera más rigurosa el rendimiento real de los algoritmos optimizados.  
Mientras que en la etapa anterior se analizó la eficiencia computacional sin validación cruzada, aquí se busca **garantizar la estabilidad, generalización y confiabilidad de los resultados** obtenidos por cada modelo.

La validación se realiza mediante la técnica de **validación cruzada estratificada (Stratified K-Fold)**, que permite dividir los datos en varios subconjuntos o *folds* manteniendo la proporción original de clases.  
Cada modelo se entrena en una combinación de estos subconjuntos y se evalúa en los restantes, repitiendo el proceso varias veces para obtener métricas promedio más representativas del desempeño general.

Durante esta etapa, se aplican los siguientes pasos:
1. Entrenamiento de los modelos previamente optimizados bajo esquemas de validación cruzada.  
2. Cálculo de métricas de evaluación como **Accuracy, Recall, F1-score y AUC**, promediadas sobre los *folds*.  
3. Comparación de los resultados validados con los obtenidos en la fase sin validación para analizar la consistencia y la capacidad de generalización.  
4. Selección de los modelos con mejor equilibrio entre rendimiento y estabilidad.



## **Objetivos**

- Validar la solidez y generalización de los modelos optimizados.  
- Identificar posibles casos de sobreajuste o subajuste.  
- Evaluar la variabilidad del rendimiento de cada modelo mediante métricas promedio y desviaciones estándar.  
- Determinar el modelo con el mejor comportamiento global y más confiable para su interpretación final.




In [1]:
# --- Manejo y análisis de datos ---
import pandas as pd
import numpy as np
import math
from collections import Counter
import time
import joblib

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocesamiento ---
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

# --- División y validación ---
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

# --- Modelos supervisados ---
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# --- Manejo de desbalanceo ---
from imblearn.over_sampling import SMOTENC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# --- Métricas de evaluación ---
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    mean_absolute_error, mean_squared_error, r2_score
)

In [2]:
X_train = pd.read_csv('x_train_resampled.csv')
y_train = pd.read_csv('y_train_resampled.csv')["diabetes"]
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')["diabetes"]

In [3]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(include="object").columns

In [6]:
# === Preprocesador ===
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cat_indices = [X_train.columns.get_loc(c) for c in cat_cols]

## **Modelo de `Regresión Logística L1 y L2`**

In [11]:
pipe_logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=5000, solver='saga', class_weight='balanced', random_state=42))
])

param_grid_logreg = {
    'model__penalty': ['l1', 'l2'],
    'model__C': [0.01, 0.1, 1, 10]
}

grid_logreg = GridSearchCV(
    estimator=pipe_logreg,
    param_grid=param_grid_logreg,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_logreg.fit(X_train, y_train)
end = time.time()


print("\n VALIDACIÓN CON DATOS BALANCEADOS COMPLETADA")
print(f"Tiempo total: {round(end - start, 2)} s")
print(f" Mejor AUC promedio (CV): {grid_logreg.best_score_:.3f}")
print(f" Mejores parámetros: {grid_logreg.best_params_}")

df_resultados_logreg = pd.DataFrame(grid_logreg.cv_results_)[
    ['mean_test_score', 'param_model__penalty', 'param_model__C']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_logreg.head())




Fitting 5 folds for each of 8 candidates, totalling 40 fits

 VALIDACIÓN CON DATOS BALANCEADOS COMPLETADA
Tiempo total: 59.86 s
 Mejor AUC promedio (CV): 0.856
 Mejores parámetros: {'model__C': 10, 'model__penalty': 'l1'}


,mean_test_score,param_model__penalty,param_model__C
6,0.856031,l1,10.0
7,0.856031,l2,10.0
4,0.856028,l1,1.0
5,0.856026,l2,1.0
2,0.855967,l1,0.1


El modelo de **Regresión Logística** con **penalización L1** y un valor de **C = 10** obtuvo el **mejor AUC promedio (0.856)** durante la validación cruzada, demostrando una excelente capacidad para discriminar entre clases.  
El hecho de que las configuraciones con penalización `l2` y valores cercanos de `C` presenten resultados similares indica una **alta estabilidad del modelo** frente a variaciones en los hiperparámetros, lo que refuerza su robustez.  

Además, el incremento del AUC respecto a la fase sin validación (≈0.789) evidencia que el balanceo de clases y el ajuste de parámetros **mejoraron significativamente la sensibilidad del modelo**, especialmente hacia la clase minoritaria.  
En conclusión, la Regresión Logística validada se consolida como uno de los modelos **más confiables y equilibrados**, combinando interpretabilidad, buen desempeño y consistencia en la generalización.

In [13]:
joblib.dump(grid_logreg.best_estimator_, "modelo_logistica.pkl")

['modelo_logistica.pkl']

## **Modelo de ``KNN_NeighborsClassifier``**

In [12]:
pipe_knn = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', KNeighborsClassifier())
])

param_grid_knn = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['minkowski', 'euclidean']
}

grid_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_grid_knn,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_knn.fit(X_train, y_train)
end = time.time()

print("\nVALIDACIÓN COMPLETADA - KNN")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_knn.best_score_:.3f}")
print(f" Mejores parámetros: {grid_knn.best_params_}")

df_resultados_knn = pd.DataFrame(grid_knn.cv_results_)[
    ['mean_test_score', 'param_model__n_neighbors', 'param_model__weights', 'param_model__metric']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_knn.head())


Fitting 5 folds for each of 16 candidates, totalling 80 fits

VALIDACIÓN COMPLETADA - KNN
Tiempo: 1819.15 s
 Mejor AUC (CV): 0.917
 Mejores parámetros: {'model__metric': 'minkowski', 'model__n_neighbors': 9, 'model__weights': 'distance'}


,mean_test_score,param_model__n_neighbors,param_model__weights,param_model__metric
7,0.917363,9,distance,minkowski
15,0.917363,9,distance,euclidean
13,0.915518,7,distance,euclidean
5,0.915518,7,distance,minkowski
3,0.910312,5,distance,minkowski


El modelo **KNN** con **9 vecinos**, ponderación por **distancia** y métrica **Minkowski** alcanzó el **mejor AUC promedio (0.917)**, superando ampliamente los resultados obtenidos en la fase sin validación (AUC ≈ 0.755).  
Este incremento notable evidencia una **mejor discriminación entre clases** y un aprendizaje más equilibrado tras aplicar el balanceo de datos.

El uso de la ponderación `distance` permitió que los vecinos más cercanos tuvieran mayor influencia en las predicciones, lo cual contribuyó a mejorar tanto la precisión como la sensibilidad del modelo.  
No obstante, el **alto tiempo de ejecución (≈30 minutos)** refleja el costo computacional del KNN, especialmente con un número elevado de vecinos y un conjunto de datos balanceado.

En conclusión, el modelo **KNN validado** presenta un **rendimiento sobresaliente en AUC**, consolidándose como uno de los algoritmos con **mejor capacidad de clasificación y generalización**, aunque con una mayor demanda de recursos computacionales.

In [14]:
joblib.dump(grid_knn.best_estimator_, 'modelo_knn.pkl')

['modelo_knn.pkl']

## **Modelo de ``Naive Bayes``**

In [7]:
pipe_nb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', GaussianNB())
])

param_grid_nb = {
    'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
}

grid_nb = GridSearchCV(
    estimator=pipe_nb,
    param_grid=param_grid_nb,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_nb.fit(X_train, y_train)
end = time.time()

print("\nVALIDACIÓN COMPLETADA - NAIVE BAYES")
print(f"Tiempo: {round(end - start, 2)} s")
print(f"Mejor AUC (CV): {grid_nb.best_score_:.3f}")
print(f"Mejores parámetros: {grid_nb.best_params_}")

df_resultados_nb = pd.DataFrame(grid_nb.cv_results_)[
    ['mean_test_score', 'param_model__var_smoothing']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_nb.head())


Fitting 5 folds for each of 4 candidates, totalling 20 fits

VALIDACIÓN COMPLETADA - NAIVE BAYES
Tiempo: 21.32 s
Mejor AUC (CV): 0.788
Mejores parámetros: {'model__var_smoothing': 1e-06}


,mean_test_score,param_model__var_smoothing
3,0.788252,1.000000e-06
2,0.788251,1.000000e-07
1,0.788251,1.000000e-08
0,0.788251,1.000000e-09


El modelo **Gaussian Naive Bayes** obtuvo un **AUC promedio de 0.788**, mostrando un desempeño consistente y prácticamente idéntico entre todas las configuraciones de `var_smoothing`.  
Esto demuestra que el modelo es **robusto frente a pequeños cambios en la suavización de varianza**, manteniendo un comportamiento estable y predecible.

Aunque el AUC es menor en comparación con modelos más complejos (como KNN o XGBoost), su **bajo costo computacional (≈21 s)** y su **simplicidad de implementación** lo convierten en una opción eficiente para tareas de clasificación rápida o como modelo base (*baseline*).  
En conclusión, **Naive Bayes validado** ofrece **buen equilibrio entre rendimiento y eficiencia**, siendo ideal para contextos donde la interpretabilidad y la velocidad son prioritarias.

In [15]:
joblib.dump(grid_nb.best_estimator_, 'modelo_nb.pkl')

['modelo_nb.pkl']

## **Modelo de ``DecisionTreeClassifier``**

In [10]:

pipe_tree = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

param_grid_tree = {
    'model__max_depth': [3, 5, 7, 10],
    'model__criterion': ['gini', 'entropy']
}

grid_tree = GridSearchCV(
    estimator=pipe_tree,
    param_grid=param_grid_tree,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_tree.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - DECISION TREE")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_tree.best_score_:.3f}")
print(f" Mejores parámetros: {grid_tree.best_params_}")

df_resultados_tree = pd.DataFrame(grid_tree.cv_results_)[
    ['mean_test_score', 'param_model__max_depth', 'param_model__criterion']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_tree.head())


Fitting 5 folds for each of 8 candidates, totalling 40 fits

 VALIDACIÓN COMPLETADA - DECISION TREE
Tiempo: 38.44 s
 Mejor AUC (CV): 0.855
 Mejores parámetros: {'model__criterion': 'gini', 'model__max_depth': 10}


,mean_test_score,param_model__max_depth,param_model__criterion
3,0.855361,10,gini
7,0.853426,10,entropy
2,0.811810,7,gini
6,0.811081,7,entropy
5,0.783702,5,entropy


El modelo **Decision Tree** con una profundidad máxima de **10** y criterio **gini** alcanzó el **mejor AUC promedio (0.855)**, evidenciando una mejora considerable respecto al AUC obtenido en la fase sin validación (≈0.749).  
Este aumento refleja una **mayor capacidad del modelo para discriminar entre clases** tras aplicar balanceo y validación cruzada, lo que sugiere una estructura más estable y generalizable.

Las configuraciones más simples (profundidades menores) mostraron una caída en el AUC, lo que indica que un árbol con mayor profundidad logra capturar mejor la complejidad del problema sin sobreajustarse excesivamente.  
En resumen, el modelo **Decision Tree validado** presenta un **rendimiento sólido, interpretable y bien equilibrado**, siendo una opción confiable cuando se busca explicabilidad sin un alto costo computacional.

In [16]:
joblib.dump(grid_tree.best_estimator_, 'modelo_tree.pkl')

['modelo_tree.pkl']

## **Modelo de ``RandomForestClassifier``**

In [ ]:
pipe_rf = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_rf.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - RANDOM FOREST")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_rf.best_score_:.3f}")
print(f" Mejores parámetros: {grid_rf.best_params_}")

df_resultados_rf = pd.DataFrame(grid_rf.cv_results_)[
    ['mean_test_score', 'param_model__n_estimators', 'param_model__max_depth', 'param_model__min_samples_split']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_rf.head())



Fitting 5 folds for each of 18 candidates, totalling 90 fits

 VALIDACIÓN COMPLETADA - RANDOM FOREST
Tiempo: 3347.74 s
 Mejor AUC (CV): 0.969
 Mejores parámetros: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 300}


,mean_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split
14,0.968545,300,None,2
13,0.968209,200,None,2
12,0.967534,100,None,2
17,0.965744,300,None,5
16,0.965402,200,None,5


['modelo_rf_cv.pkl']

El modelo **Random Forest** alcanzó un **AUC promedio sobresaliente de 0.969**, posicionándose como uno de los **mejores modelos validados** en términos de capacidad predictiva.  
La combinación de **300 árboles** sin restricción de profundidad permitió una alta flexibilidad y una excelente discriminación entre clases, sin evidenciar sobreajuste significativo gracias al proceso de validación cruzada.

El incremento en el AUC respecto a la fase sin validación (≈0.795) demuestra una mejora sustancial en la **generalización del modelo** y en su capacidad para adaptarse a los datos balanceados.  
El costo computacional fue elevado (≈55 minutos), lo cual es esperable en ensambles grandes, pero se justifica por el rendimiento obtenido.

En resumen, el **Random Forest validado** se consolida como **uno de los modelos más robustos, estables y precisos**, logrando una excelente relación entre rendimiento predictivo y fiabilidad, aunque con un alto requerimiento de tiempo y recursos computacionales.

In [18]:
joblib.dump(grid_rf.best_estimator_, 'modelo_rf.pkl')

['modelo_rf.pkl']

## **Modelo de ``XGBClassifier``**

In [9]:
pipe_xgb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

param_grid_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1]
}

grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_xgb.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - XGBOOST")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_xgb.best_score_:.3f}")
print(f" Mejores parámetros: {grid_xgb.best_params_}")

df_resultados_xgb = pd.DataFrame(grid_xgb.cv_results_)[
    ['mean_test_score', 'param_model__n_estimators', 'param_model__max_depth', 'param_model__learning_rate']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_xgb.head())


Fitting 5 folds for each of 27 candidates, totalling 135 fits

 VALIDACIÓN COMPLETADA - XGBOOST
Tiempo: 294.67 s
 Mejor AUC (CV): 0.966
 Mejores parámetros: {'model__learning_rate': 0.1, 'model__max_depth': 8, 'model__n_estimators': 300}


,mean_test_score,param_model__n_estimators,param_model__max_depth,param_model__learning_rate
26,0.966129,300,8,0.10
25,0.965369,200,8,0.10
23,0.964820,300,6,0.10
17,0.964584,300,8,0.05
22,0.963561,200,6,0.10


El modelo **XGBoost** alcanzó un **AUC promedio de 0.966**, confirmando su **excelente capacidad de clasificación y generalización** tras el proceso de validación cruzada.  
La combinación de **300 árboles**, **profundidad moderada (8)** y una **tasa de aprendizaje media (0.1)** permitió un equilibrio óptimo entre precisión y estabilidad, maximizando el poder predictivo sin sobreajustar los datos.

En comparación con la fase sin validación (AUC ≈ 0.805), el incremento de más de 16 puntos evidencia una **mejora notable en la discriminación de clases**, gracias al balanceo de datos y al ajuste fino de hiperparámetros.  
Además, el tiempo de ejecución (≈5 minutos) fue razonable considerando la complejidad del modelo y la cantidad de árboles utilizados.

En conclusión, el **XGBoost validado** se posiciona como **uno de los modelos más potentes del estudio**, destacando por su alto rendimiento, eficiencia y capacidad de generalización, con un excelente equilibrio entre complejidad y desempeño.

In [13]:
joblib.dump(grid_xgb.best_estimator_, 'modelo_xgb.pkl')

['modelo_xgb.pkl']

## **Modelo de ``LinearSVC``**

In [ ]:
pipe_svm = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', CalibratedClassifierCV(
        estimator=LinearSVC(max_iter=5000, class_weight='balanced', random_state=42),
        cv=3
    ))
])

param_grid_svm = {
    'model__estimator__C': [0.01, 0.1, 1, 10]   #  parámetro actualizado
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_svm = GridSearchCV(
    estimator=pipe_svm,
    param_grid=param_grid_svm,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_svm.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - LINEAR SVC (Calibrado)")
print(f"Tiempo total: {round(end - start, 2)} s")
print(f" Mejor AUC promedio (CV): {grid_svm.best_score_:.3f}")
print(f" Mejores parámetros: {grid_svm.best_params_}")

df_resultados_svm = pd.DataFrame(grid_svm.cv_results_)[
    ['mean_test_score', 'param_model__estimator__C']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_svm.head())



Fitting 5 folds for each of 4 candidates, totalling 20 fits

 VALIDACIÓN COMPLETADA - LINEAR SVC (Calibrado)
Tiempo total: 54.43 s
 Mejor AUC promedio (CV): 0.856
 Mejores parámetros: {'model__estimator__C': 10}


,mean_test_score,param_model__estimator__C
3,0.855719,10.00
2,0.855715,1.00
1,0.855680,0.10
0,0.855307,0.01


El modelo **Linear SVC calibrado** con **C = 10** alcanzó un **AUC promedio de 0.856**, mostrando un desempeño robusto y estable en la clasificación de las clases tras el proceso de validación cruzada.  
Las diferencias mínimas entre los distintos valores de `C` indican una **alta estabilidad del modelo**, lo que demuestra que su rendimiento no depende fuertemente de pequeñas variaciones en el grado de regularización.

El incremento del AUC respecto a la fase sin validación (≈0.789) evidencia una **mejora sustancial en la capacidad de discriminación**, atribuida tanto al balanceo de los datos como al uso de probabilidades calibradas.  
Además, el tiempo de validación fue razonable, manteniendo una buena relación entre **eficiencia computacional y calidad predictiva**.

En conclusión, el **Linear SVC calibrado validado** se consolida como un modelo **lineal altamente estable y confiable**, ofreciendo un rendimiento competitivo frente a algoritmos más complejos, pero con la ventaja adicional de una **interpretabilidad y eficiencia superiores**.

In [20]:

joblib.dump(grid_svm.best_estimator_, 'modelo_svm.pkl')

['modelo_svm.pkl']